# Attention Steering via Symmetric-Antisymmetric Decomposition

This notebook explores the decomposition of attention matrices into:
- **Symmetric component S = (A + Aᵀ)/2** — real eigenvalues, captures mutual/bidirectional attention
- **Antisymmetric component K = (A - Aᵀ)/2** — imaginary eigenvalues, captures directional information flow

We then use this decomposition to **steer** model behavior by modifying the spectral properties of each component.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from attention_steering import AttentionDecomposer, AttentionExtractor, AttentionSteerer
from attention_steering.steer import SteeringConfig
from attention_steering.viz import (
    plot_attention_decomposition,
    plot_eigenvalue_spectra,
    plot_asymmetry_across_layers,
    plot_steering_comparison,
    plot_eigenvalue_evolution,
)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Extract Attention Maps

Works with both text-only LLMs and Vision-Language Models (VLMs):
- Text-only: `AttentionExtractor("gpt2")`, `AttentionExtractor("meta-llama/Llama-3.2-1B")`
- VLM: `AttentionExtractor("Qwen/Qwen2.5-VL-3B-Instruct")`

In [ ]:
# --- Choose your model ---
# Text-only LLM (small, runs on CPU):
extractor = AttentionExtractor("gpt2")

# Vision-Language Model with 4-bit quantization (~3GB VRAM, fits on free Colab T4):
# extractor = AttentionExtractor("Qwen/Qwen2.5-VL-3B-Instruct", load_in_4bit=True)

# Other loading options:
#   load_in_8bit=True            — 8-bit quantization (~4.5GB)
#   torch_dtype=torch.float16    — half precision, no quantization (~7GB)
#   device_map="auto"            — automatic CPU/GPU split for very large models

print(extractor.get_model_info())

In [ ]:
text = "The cat sat on the mat and looked at the bird"

# For text-only models:
attn_maps = extractor.extract(text, return_tokens=True)

# For VLMs with an image (uncomment):
# attn_maps = extractor.extract(
#     "Describe this image in detail",
#     images=["https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/300px-PNG_transparency_demonstration_1.png"],
#     return_tokens=True,
# )

print(attn_maps)
print(f"Tokens: {attn_maps.tokens}")

## 2. Symmetric-Antisymmetric Decomposition

For any attention matrix **A**, we decompose:
- $S = \frac{A + A^T}{2}$ — symmetric, guaranteed real eigenvalues
- $K = \frac{A - A^T}{2}$ — skew-symmetric, guaranteed purely imaginary eigenvalues

And $A = S + K$ exactly.

In [ ]:
decomposer = AttentionDecomposer()

# Pick a specific layer and head to examine
layer, head = 5, 0
A = attn_maps.get(layer, head)

S, K = decomposer.decompose(A)

# Verify the decomposition
reconstruction_error = (A - decomposer.reconstruct(S, K)).abs().max().item()
print(f"Reconstruction error: {reconstruction_error:.2e}")

# Verify symmetry properties
sym_error = (S - S.T).abs().max().item()
skew_error = (K + K.T).abs().max().item()
print(f"Symmetry check on S (should be ~0): {sym_error:.2e}")
print(f"Skew-symmetry check on K (should be ~0): {skew_error:.2e}")

In [ ]:
# Visualize the decomposition
fig = plot_attention_decomposition(
    A, tokens=attn_maps.tokens,
    title=f"Layer {layer}, Head {head}"
)
plt.show()

## 3. Eigenvalue Spectra Analysis

The key insight:
- **S** has real eigenvalues → stable, energy-like modes ("what tokens agree on")
- **K** has imaginary eigenvalues → rotational/oscillatory modes ("directional information flow")

In [ ]:
fig = plot_eigenvalue_spectra(
    A,
    title=f"Eigenvalue Spectra — Layer {layer}, Head {head}"
)
plt.show()

In [ ]:
# Numerical eigenvalue analysis
real_eigs = decomposer.eigenspectrum_symmetric(S)
imag_eigs = decomposer.eigenspectrum_antisymmetric(K)

print("=== Symmetric (Real) Eigenvalues ===")
print(f"  Top 5: {real_eigs[:5].numpy()}")
print(f"  Sum: {real_eigs.sum():.4f} (trace of S)")
print(f"  Energy (sum of squares): {(real_eigs**2).sum():.4f}")

print("\n=== Antisymmetric (Imaginary) Eigenvalues ===")
print(f"  Top 5 (by magnitude): {imag_eigs[:5].numpy()}")
print(f"  Sum: {imag_eigs.sum():.6f} (should be ~0)")
print(f"  Energy (sum of squares): {(imag_eigs**2).sum():.4f}")

## 4. Asymmetry Analysis Across Layers and Heads

The **asymmetry score** measures the ratio of antisymmetric energy to total energy:

$$\text{score} = \frac{\|K\|_F^2}{\|S\|_F^2 + \|K\|_F^2}$$

- Score ≈ 0 → attention is nearly symmetric (bidirectional)
- Score ≈ 1 → attention is highly directional

In [ ]:
fig = plot_asymmetry_across_layers(attn_maps.attentions)
plt.show()

## 5. Eigenvalue Evolution Across Layers

Track how the top eigenvalues change as information flows through layers.

In [ ]:
# Track symmetric eigenvalues for head 0 across all layers
head_attns = attn_maps.get_head_across_layers(0)

fig = plot_eigenvalue_evolution(
    head_attns, component="symmetric", top_k=5,
)
plt.show()

fig = plot_eigenvalue_evolution(
    head_attns, component="antisymmetric", top_k=5,
)
plt.show()

## 6. Spectral Filtering — Low-Rank Approximation

Keep only the top-k eigenvalues of each component to see which modes carry the most information.

In [ ]:
# Compare original with spectrally filtered versions
for top_k in [1, 3, 5]:
    S_filtered = decomposer.spectral_filter_symmetric(S, top_k)
    K_filtered = decomposer.spectral_filter_antisymmetric(K, top_k)
    A_filtered = decomposer.reconstruct(S_filtered, K_filtered)

    error = (A - A_filtered).norm() / A.norm()
    print(f"top_k={top_k}: relative reconstruction error = {error:.4f}")

    fig = plot_steering_comparison(
        A, A_filtered, tokens=attn_maps.tokens,
        title=f"Spectral Filtering (top_k={top_k})"
    )
    plt.show()

## 7. Steering Experiments

Now we steer attention by scaling the eigenvalues of S and K independently.

In [ ]:
# Experiment: scale symmetric vs antisymmetric components
configs = [
    ("Boost symmetric (1.5x)", 1.5, 1.0),
    ("Dampen symmetric (0.5x)", 0.5, 1.0),
    ("Boost antisymmetric (1.5x)", 1.0, 1.5),
    ("Dampen antisymmetric (0.5x)", 1.0, 0.5),
    ("Boost sym + dampen asym", 1.5, 0.5),
    ("Dampen sym + boost asym", 0.5, 1.5),
]

for name, sym_scale, asym_scale in configs:
    A_steered = decomposer.steer(
        A, symmetric_scale=sym_scale, antisymmetric_scale=asym_scale
    )
    fig = plot_steering_comparison(
        A, A_steered, tokens=attn_maps.tokens, title=name
    )
    plt.show()

## 8. Generation with Steering Hooks

Apply steering during actual text generation and compare outputs.

In [ ]:
# from_extractor reuses the loaded model — works for both LLMs and VLMs
steerer = AttentionSteerer.from_extractor(extractor)

prompt = "The future of artificial intelligence is"

# Baseline (no steering)
baseline = steerer.generate_with_steering(prompt, max_new_tokens=40)
print("=== Baseline ===")
print(baseline['full_text'])
print()

In [ ]:
# Steering: boost mutual attention, suppress directional flow
config_sym = SteeringConfig(
    layers=[4, 5, 6, 7, 8],
    symmetric_scale=1.5,
    antisymmetric_scale=0.3,
)
result_sym = steerer.generate_with_steering(prompt, config=config_sym, max_new_tokens=40)
print("=== Boosted Symmetric / Dampened Antisymmetric ===")
print(result_sym['full_text'])
print(f"\nSteering log: {result_sym['steering_log']}")

In [ ]:
# Steering: suppress mutual attention, boost directional flow
config_asym = SteeringConfig(
    layers=[4, 5, 6, 7, 8],
    symmetric_scale=0.5,
    antisymmetric_scale=2.0,
)
result_asym = steerer.generate_with_steering(prompt, config=config_asym, max_new_tokens=40)
print("=== Dampened Symmetric / Boosted Antisymmetric ===")
print(result_asym['full_text'])
print(f"\nSteering log: {result_asym['steering_log']}")

In [ ]:
# Full comparison across multiple prompts
prompts = [
    "The cat sat on the",
    "In the beginning, there was",
    "The most important thing about science is",
]

config = SteeringConfig(
    layers=[5, 6, 7],
    symmetric_scale=1.5,
    antisymmetric_scale=0.5,
)

for p in prompts:
    result = steerer.compare_steered_vs_baseline(p, config, max_new_tokens=30)
    print(f"Prompt: '{result['prompt']}'")
    print(f"  Baseline:  {result['baseline']}")
    print(f"  Steered:   {result['steered']}")
    print()

## 9. Custom Steering Transform

You can define arbitrary transforms on (S, K) for more fine-grained control.

In [ ]:
def zero_antisymmetric(S, K):
    """Remove all directional asymmetry — force purely symmetric attention."""
    return S, torch.zeros_like(K)

def zero_symmetric(S, K):
    """Remove all mutual attention — keep only directional flow."""
    return torch.zeros_like(S), K

# Test with custom transform
config_custom = SteeringConfig(
    layers=[5, 6, 7],
    custom_transform=zero_antisymmetric,
)

prompt = "Once upon a time in a land far away"
result = steerer.compare_steered_vs_baseline(prompt, config_custom, max_new_tokens=40)
print(f"Prompt: '{result['prompt']}'")
print(f"  Baseline (normal):    {result['baseline']}")
print(f"  Steered (sym only):   {result['steered']}")